In [ ]:

from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from src.unit_proccessing import  *
from src.utils.stats_utils import *
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
with initialize(config_path='../configuration', version_base='1.1'):
    config = compose(config_name='main.yaml')

In [ ]:
features_class = UnitDataProcessing(config)
self = features_class

# Answer Time Changed UNIT Level Processing

##### Get Feature and process

In [ ]:
feature_name = 'f__number_answered'
score_name = self.rename_feature(feature_name)
df = self.df_unit[~pd.isnull(self.df_unit[feature_name])]

In [ ]:
df[feature_name].hist()

In [ ]:
contamination = self.get_contamination_parameter(feature_name, method='medfilt', random_state=42)

model = ECOD(contamination=0.11)
model.fit(self._df_unit[[feature_name]])
self._df_unit[score_name] = model.predict(self._df_unit[[feature_name]])

score_name1 = score_name + '_lower'
score_name2 = score_name + '_upper'
min_good_value = self._df_unit[(self._df_unit[score_name] == 0)][feature_name].min()
max_good_value = self._df_unit[(self._df_unit[score_name] == 0)][feature_name].max()

self._df_unit[score_name1] = 0
self._df_unit[score_name2] = 0

self._df_unit.loc[(self._df_unit[feature_name] < min_good_value), score_name1] = 1
self._df_unit.loc[(self._df_unit[feature_name] > max_good_value), score_name2] = 1

In [ ]:
bins = np.histogram_bin_edges(df[feature_name], bins=48)
data_true = df[df[score_name]==0][feature_name]
data_false = df[df[score_name]==1][feature_name]

plt.hist(data_true, bins=bins, alpha=0.5, color='blue', label='True')
plt.hist(data_false, bins=bins, alpha=0.5, color='red', label='False')
plt.show()

In [ ]:
##########################################################################################################################################################